# CRNN (Small) — Convolutional Recurrent Neural Network for Text Recognition

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Ffrom torch.utils.data import Dataset, DataLoader

In [ ]:
class Config:    img_height = 32    img_width = 100    n_channels = 1    num_classes = 37    hidden_size = 256    max_label_len = 10cfg = Config()device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class CNNBackbone(nn.Module):    def __init__(self, cfg):        super().__init__()        self.layers = nn.Sequential(            nn.Conv2d(cfg.n_channels, 64, 3, 1, 1),            nn.ReLU(inplace=True),            nn.MaxPool2d(2, 2),            nn.Conv2d(64, 128, 3, 1, 1),            nn.ReLU(inplace=True),            nn.MaxPool2d(2, 2),            nn.Conv2d(128, 256, 3, 1, 1),            nn.BatchNorm2d(256),            nn.ReLU(inplace=True),            nn.Conv2d(256, 256, 3, 1, 1),            nn.ReLU(inplace=True),            nn.MaxPool2d((2, 1), (2, 1)),            nn.Conv2d(256, 512, 3, 1, 1),            nn.BatchNorm2d(512),            nn.ReLU(inplace=True),            nn.Conv2d(512, 512, 3, 1, 1),            nn.ReLU(inplace=True),            nn.MaxPool2d((2, 1), (2, 1)),            nn.Conv2d(512, 512, 2, 1, 0),            nn.BatchNorm2d(512),            nn.ReLU(inplace=True),        )    def forward(self, x):        return self.layers(x)

In [ ]:
class BidirectionalLSTM(nn.Module):    def __init__(self, input_size, hidden_size, output_size):        super().__init__()        self.rnn = nn.LSTM(input_size, hidden_size, bidirectional=True, batch_first=True)        self.linear = nn.Linear(hidden_size * 2, output_size)    def forward(self, x):        recurrent, _ = self.rnn(x)        return self.linear(recurrent)

In [ ]:
class CRNN(nn.Module):    def __init__(self, cfg):        super().__init__()        self.cnn = CNNBackbone(cfg)        self.rnn = nn.Sequential(            BidirectionalLSTM(512, cfg.hidden_size, cfg.hidden_size),            BidirectionalLSTM(cfg.hidden_size, cfg.hidden_size, cfg.num_classes),        )    def forward(self, x):        conv = self.cnn(x)        b, c, h, w = conv.size()        conv = conv.squeeze(2)        conv = conv.permute(0, 2, 1)        logits = self.rnn(conv)        return logits

In [ ]:
class ToyOCRDataset(Dataset):    def __init__(self, n_samples, cfg):        self.n_samples = n_samples        self.cfg = cfg    def __len__(self):        return self.n_samples    def __getitem__(self, idx):        image = torch.randn(self.cfg.n_channels, self.cfg.img_height, self.cfg.img_width)        label_len = torch.randint(3, self.cfg.max_label_len, (1,)).item()        label = torch.randint(1, self.cfg.num_classes, (label_len,))        return image, labeldef collate_fn(batch):    images, labels = zip(*batch)    image_batch = torch.stack(images)    label_lengths = torch.tensor([label.size(0) for label in labels], dtype=torch.long)    max_label_len = max(label_lengths).item()    label_batch = torch.zeros(len(batch), max_label_len, dtype=torch.long)    for i, label in enumerate(labels):        label_batch[i, :label.size(0)] = label    return image_batch, label_batch, label_lengths

In [ ]:
dataset = ToyOCRDataset(n_samples=64, cfg=cfg)loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)model = CRNN(cfg).to(device)optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)

In [ ]:
def train_epoch(model, loader, optimizer, ctc_loss):    model.train()    total_loss = 0.0    for images, labels, label_lengths in loader:        images = images.to(device)        labels = labels.to(device)        label_lengths = label_lengths.to(device)        logits = model(images)        log_probs = F.log_softmax(logits, dim=2).permute(1, 0, 2)        input_lengths = torch.full((images.size(0),), log_probs.size(0), dtype=torch.long, device=device)        loss = ctc_loss(log_probs, labels, input_lengths, label_lengths)        optimizer.zero_grad()        loss.backward()        optimizer.step()        total_loss += loss.item()    return total_loss / len(loader)

In [ ]:
for epoch in range(5):    avg_loss = train_epoch(model, loader, optimizer, ctc_loss)    print(f"epoch {epoch + 1} loss {avg_loss:.4f}")

In [ ]:
def ctc_greedy_decode(logits, blank=0):    pred_indices = logits.argmax(dim=2)    decoded_batch = []    for sequence in pred_indices:        decoded = []        previous = None        for index in sequence.tolist():            if index != previous and index != blank:                decoded.append(index)            previous = index        decoded_batch.append(decoded)    return decoded_batch

In [ ]:
model.eval()with torch.no_grad():    sample_images = torch.randn(4, cfg.n_channels, cfg.img_height, cfg.img_width).to(device)    logits = model(sample_images)    decoded = ctc_greedy_decode(logits)    for sequence in decoded:        print(sequence)